In [ ]:
using Bessels, MPSDynamics, ChaosTools, ProgressBars
include("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/config/requirements.jl");

# Check vital signs

In [ ]:
const ∞ = Inf
ħ=1
e=1
β = ∞ # Case zero temperature T=0, β → ∞
ϕ_quantum=1/2

#### Caracterization of the circuit #####

ω_a = 3.8533*2*pi
ω_b_0 = 7.0515*2*pi

Lamb_shift = 0.05 #Lamb shift in GHz


φ_a = 0.11
Z_a = 2*ϕ_quantum^2*φ_a^2
C_a_inv = Z_a*ω_a
E_Ca = C_a_inv/2

φ_b = 0.204
Z_b = 2*ϕ_quantum^2*φ_b^2
C_b_inv = Z_b*ω_b_0
E_Cb = C_b_inv/2

E_J = 37.12*2*pi


##Drive


ω_d=ω_b_0#+Lamb_shift*2*pi # Drive frequency
ω_p=2ω_a-ω_d

ϵ_0 = 0.4581818181818182
g2 = besselj(1,ϵ_0)*E_J*φ_a^2*φ_b # (Leon's paper)
ϵ_d=10.12*g2

buffer_phase = 0.0
pump_phase = 0.0;

##Env

κ_b=1/10.4; # 1/ns from Leon's paper
κ_2 = 4*g2^2/κ_b
κ_a = 0#3e-3*κ_2 #from Leon's paper

In [ ]:
t_max=60/κ_b
re_a_0, im_a_0 = -10, 2
ϕ_a_0, q_a_0=sqrt(2*Z_a)*re_a_0, sqrt(2/Z_a)*im_a_0
ϕ_b_0, q_b_0=0.0, 0.0

system = SystemATS(ω_a, ω_b_0, E_J, E_Ca, E_Cb, ϕ_a_0, q_a_0, ϕ_b_0, q_b_0)
drive = DriveTwoModes(ϵ_d, ω_d, buffer_phase, ϵ_0, ω_p, pump_phase)
environment = MeanField(κ_a, κ_b)

ϕ_CL_a, ϕ_CL_b, q_CL_a, q_CL_b, t_list_CL = simulator(t_max, system, drive, environment);

In [ ]:
p1,p2 = plot_mode(ϕ_CL_a, q_CL_a, t_list_CL, ω_a, Z_a)

plot_perso(p1,p2,layout=(1,2),
xlabel=L"\Re(a)", ylabel=L"\Im(a)",
title=["Resonator a" "Resonator a RF"],
legend=false)

In [ ]:
p1,p2 = plot_mode(ϕ_CL_b, q_CL_b, t_list_CL, ω_b_0 , Z_b)
scatter!((1.7,-0.7))
plot_perso(p1,p2,layout=(1,2),
xlabel=L"\Re(a)", ylabel=L"\Im(a)",
title=["Resonator a" "Resonator a RF"],
legend=false)


# Lyapunov maps

In [ ]:
t_max = 10/κ_b
t_min = 0.0
dt = 1 / (50 * ω_a) # Time step for the simulation


re_b_0, im_b_0 = 0, 0
ϕ_b_0, q_b_0=sqrt(2*Z_b)*re_b_0, sqrt(2/Z_b)*im_b_0
drive = DriveTwoModesBasics(ϵ_d, ω_d, buffer_phase, ϵ_0, ω_p, pump_phase)
environment = MeanField(κ_a, κ_b)
drive_prepared = prepare(drive)
system = SystemATS(ω_a, ω_b_0, E_J, E_Ca, E_Cb, 0.0, 0.0, ϕ_b_0, q_b_0)

environment_prepared = prepare(system,environment)

In [ ]:
ϵ_0 = 0.5127
g2 = besselj(1,ϵ_0)*E_J*φ_a^2*φ_b # (Leon's paper)
ϵ_d_g2_ratio_list = 1:1:15

In [ ]:
Im_a_list=range(0,6,15)
Re_a_list=range(-15,-2,26)
Lyap_coef_dict = Dict{Float64, Array{Float64}}()
t_w = 30/κ_b
for ϵ_d_g2_ratio_i in ProgressBar(ϵ_d_g2_ratio_list)
    ϵ_d_i=ϵ_d_g2_ratio_i*g2
    drive = DriveTwoModesBasics(ϵ_d_i, ω_d, buffer_phase, ϵ_0, ω_p, pump_phase)
    drive_prepared = prepare(drive)
    Lyap_coef = Array{Float64}(undef, length(Im_a_list), length(Re_a_list))

    for i in 1:length(Re_a_list)
        re_i = Re_a_list[i]
        ϕ_a_0_i=sqrt(2*Z_a)*re_i
        for j in 1:length(Im_a_list)
            im_j = Im_a_list[j]
            q_a_0_j = sqrt(2/Z_a)*im_j

            system = SystemATS(ω_a, ω_b_0, E_J, E_Ca, E_Cb, ϕ_a_0_i, q_a_0_j, ϕ_b_0, q_b_0)
            system_prepared = prepare(system)

            u_dot_0, u_0 = get_init(system_prepared, environment_prepared)
            u_0_new = vcat(u_0,u_dot_0);
            
            p = SimulationParams(system_prepared, drive_prepared, environment_prepared)

            problem = CoupledODEs(equation_1ODE!, u_0_new, p)

            Lyap_coef[j,i] = lyapunov(problem, t_max, Δt=dt; d0 = 1e-7, d0_upper = 1e-4, Ttr = t_w)
        end
    end
    Lyap_coef_dict[ϵ_d_g2_ratio_i] = Lyap_coef
end

In [ ]:
using JSON
Lyap_coef_strkeys = Dict(string(k) => reduce(vcat,v) for (k,v) in Lyap_coef_dict)
open("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/analysis/ouput/data_lyapunov_full_sine_sweep_alpha_28_11_2025.json", "w") do io
    JSON.print(io, Lyap_coef_strkeys)
end

In [ ]:
for ϵ_d_g2_ratio_i in ϵ_d_g2_ratio_list
    Lyap_coef = Lyap_coef_dict[ϵ_d_g2_ratio_i]
    p0 = heatmap(Re_a_list,Im_a_list,Lyap_coef)

    plot!(xlabel=L"\Re(a)", ylabel=L"\Im(a)",
        size=(size_px_car[1],size_px_car[2]),
        xlims=(minimum(Re_a_list),maximum(Re_a_list)),ylims=(minimum(Im_a_list),maximum(Im_a_list)),
        colorbar_title="\nLyapunov coef.", title = "Mean field"*"\n"*L"κ_b t= "*"$(κ_b*t_max)"*", "*L"\varepsilon_d/g_2= "*string(round(ϵ_d_g2_ratio_i,digits=3))*" MHz",
        right_margin=10mm)
    display(plot(p0,
    xlims=(minimum(Re_a_list),maximum(Re_a_list)),
    ylims=(minimum(Im_a_list),maximum(Im_a_list)),top_margin=5mm))
end

In [ ]:
Lyap_coef_dict

In [ ]:
Im_a_list=range(-8,8,80)
Re_a_list=range(-20,20,240)

ϵ_0 = 0.64
g2 = besselj(1,ϵ_0)*E_J*φ_a^2*φ_b # (Leon's paper)
ϵ_d=10*g2
drive = DriveTwoModes(ϵ_d, ω_d, buffer_phase, ϵ_0, ω_p, pump_phase)
drive_prepared = prepare(drive)
Lyap_coef = Array{Float64}(undef, length(Im_a_list), length(Re_a_list))

for i in ProgressBar(1:length(Re_a_list))
    re_i = Re_a_list[i]
    ϕ_a_0_i=sqrt(2*Z_a)*re_i
    for j in 1:length(Im_a_list)
        im_j = Im_a_list[j]
        q_a_0_j = sqrt(2/Z_a)*im_j

        system = SystemATS(ω_a, ω_b_0, E_J, E_Ca, E_Cb, ϕ_a_0_i, q_a_0_j, ϕ_b_0, q_b_0)
        system_prepared = prepare(system)

        u_dot_0, u_0 = get_init(system_prepared, environment_prepared)
        u_0_new = vcat(u_0,u_dot_0);

        p = SimulationParams(system_prepared, drive_prepared, environment_prepared)

        problem = CoupledODEs(equation_1ODE!, u_0_new, p)

        Lyap_coef[j,i] = lyapunov(problem, t_max, Δt=dt; d0 = 1e-7, d0_upper = 1e-4, Ttr = 100*dt)
    end
end
p0 = heatmap(Re_a_list,Im_a_list,Lyap_coef)

plot!(xlabel=L"\Re(a)", ylabel=L"\Im(a)",
    size=(size_px_car[1],size_px_car[2]),
    xlims=(minimum(Re_a_list),maximum(Re_a_list)),ylims=(minimum(Im_a_list),maximum(Im_a_list)),
    colorbar_title="\nLyapunov coef.", title = "Mean field"*"\n"*L"κ_b t= "*"$(κ_b*t_max)"*", "*L"g_2/2\pi= "*string(round(g2/2pi*1000,digits=3))*" MHz",
    clims=(0,0.35),
    right_margin=10mm)
display(plot(p0,
xlims=(minimum(Re_a_list),maximum(Re_a_list)),
ylims=(minimum(Im_a_list),maximum(Im_a_list)),top_margin=5mm))


In [ ]:
ϵ_0 = 0.5473684210526316
g2_max = besselj(1,ϵ_0)*E_J*φ_a^2*φ_b # (Leon's paper)
ϵ_d=10*g2_max

buffer_phase = 0.0
pump_phase = 0.0;

##Env

κ_b=1/10; # 1/ns from Leon's paper
κ_2 = 4*g2_max^2/κ_b
κ_a = 3e-3*κ_2 #from Leon's paper

In [ ]:
t_max=10/κ_b
ϕ_b_0, q_b_0=0.0, 0.0
drive = DriveTwoModes(ϵ_d, ω_d, buffer_phase, ϵ_0, ω_p, pump_phase)
environment = MeanField(κ_a, κ_b)
drive_prepared = prepare(drive)
system = SystemATS(ω_a, ω_b_0, E_J, E_Ca, E_Cb, 0.0, 0.0, ϕ_b_0, q_b_0)


environment_prepared = prepare(system,environment)

dt = get_dt(system, environment)

In [ ]:
Im_a_list=range(-8,8,80)
Re_a_list=range(-10,10,80)
Lyap_coef = Array{Float64}(undef, length(Im_a_list), length(Re_a_list))

for i in ProgressBar(1:length(Re_a_list))
    re_i = Re_a_list[i]
    ϕ_a_0_i=sqrt(2*Z_a)*re_i
    for j in 1:length(Im_a_list)
        im_j = Im_a_list[j]
        q_a_0_j = sqrt(2/Z_a)*im_j

        system = SystemATS(ω_a, ω_b_0, E_J, E_Ca, E_Cb, ϕ_a_0_i, q_a_0_j, ϕ_b_0, q_b_0)
        system_prepared = prepare(system)

        u_dot_0, u_0 = get_init(system_prepared, environment_prepared)
        u_0_new = vcat(u_0,u_dot_0);

        p = SimulationParams(system_prepared, drive_prepared, environment_prepared)

        problem = CoupledODEs(equation_1ODE!, u_0_new, p)

        Lyap_coef[j,i] = lyapunov(problem, t_max, Δt=dt; d0 = 1e-7, d0_upper = 1e-4, Ttr = 100*dt)
    end
end

In [ ]:
heatmap(Re_a_list,Im_a_list,Lyap_coef)

plot!(xlabel=L"\Re(a)", ylabel=L"\Im(a)",
    size=(size_px_car[1],size_px_car[2]),
    xlims=(minimum(Re_a_list),maximum(Re_a_list)),ylims=(minimum(Im_a_list),maximum(Im_a_list)),
    colorbar_title="\nLyapunov coef.", title = "Mean field "*L"κ_b t= "*"$(κ_b*t_max)",
    clims=(0,maximum(Lyap_coef)),
    right_margin=10mm)
#savefig("../../Figures/Two-photon/heatmap_lyapunov_coef.pdf")